In [6]:
"""
Streaming Language Modeling Data Pipeline with Hugging Face Datasets
--------------------------------------------------------------------
Goal:
    Demonstrate how to build a *true streaming* LM pipeline that:
    - Processes data without loading the entire dataset into RAM.
    - Tokenizes on the fly.
    - Concatenates text and chunks into fixed-length blocks for LM training.
    - Produces batches ready for training in PyTorch.

Key Teaching Points:
    1. Streaming allows us to work with web-scale corpora.
    2. We still can do grouping/chunking in a rolling fashion.
    3. This approach mimics real-world pipelines for large-scale LM training.
"""
print(__doc__)


Streaming Language Modeling Data Pipeline with Hugging Face Datasets
--------------------------------------------------------------------
Goal:
    Demonstrate how to build a *true streaming* LM pipeline that:
    - Processes data without loading the entire dataset into RAM.
    - Tokenizes on the fly.
    - Concatenates text and chunks into fixed-length blocks for LM training.
    - Produces batches ready for training in PyTorch.

Key Teaching Points:
    1. Streaming allows us to work with web-scale corpora.
    2. We still can do grouping/chunking in a rolling fashion.
    3. This approach mimics real-world pipelines for large-scale LM training.



In [7]:
!pip install transformers datasets torch

In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch

In [9]:
#  1. Load the dataset in STREAMING mode
# Streaming mode returns an IterableDataset — you can iterate over it
# without having all the data in memory at once.
stream_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)
print("Dataset loaded in streaming mode:", stream_dataset)

Dataset loaded in streaming mode: IterableDataset({
    features: ['text'],
    num_shards: 4
})


In [11]:
# 2. Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"PAD token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

Vocab size: 30522
PAD token: '[PAD]' (id=0)


In [12]:
# 3. Tokenization step
# We do NOT pad/truncate here — we want raw token sequences.
# This keeps flexibility to later concatenate across documents.
def tokenize_function(examples):
    return tokenizer(examples["text"], add_special_tokens=False)
 
# Map tokenization lazily over the streaming dataset
tokenized_stream = stream_dataset.map(tokenize_function, batched=True)

In [13]:
# 4. Rolling buffer for grouping into fixed-length blocks
# Because streaming datasets are iterators, we can't look ahead arbitrarily.
# We'll keep a buffer that stores leftover tokens from the previous batch,
# so we can concatenate and chunk consistently.
block_size = 128
 
def group_texts_streaming(dataset_iter, block_size):
    buffer = []
    for example in dataset_iter:
        buffer.extend(example["input_ids"])
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {
                "input_ids": chunk,
                "attention_mask": [1] * block_size
            }

In [14]:
# 5. Wrap generator in an IterableDataset
class StreamingLMIterableDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, block_size):
        self.dataset = hf_iterable_dataset
        self.block_size = block_size
 
    def __iter__(self):
        return group_texts_streaming(self.dataset, self.block_size)
 
grouped_iterable_dataset = StreamingLMIterableDataset(tokenized_stream, block_size)

In [15]:
# 6. Collate function for batches
def collate_fn(batch):
    input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()
    }

In [16]:
# 7. DataLoader for streaming data
train_loader = DataLoader(grouped_iterable_dataset, batch_size=8, collate_fn=collate_fn)

In [17]:
# 8. Iterate over a few batches
print("Sample streaming batches:")
for i, batch in enumerate(train_loader):
    print(f"Batch {i} -> input_ids shape: {batch['input_ids'].shape}")
    if i == 2:
        break

Sample streaming batches:


Token indices sequence length is longer than the specified maximum sequence length for this model (948 > 512). Running this sequence through the model will result in indexing errors


Batch 0 -> input_ids shape: torch.Size([8, 128])
Batch 1 -> input_ids shape: torch.Size([8, 128])
Batch 2 -> input_ids shape: torch.Size([8, 128])


In [19]:
# 9. Token-ID distribution analysis on one batch
import collections
 
sample_batch = next(iter(train_loader))
flat_ids = sample_batch["input_ids"].flatten().tolist()
 
counter = collections.Counter(flat_ids)
top10 = counter.most_common(10)
 
print(f"Total tokens in batch : {len(flat_ids)}")
print(f"Unique token IDs : {len(counter)}")
print("\nTop-10 most frequent token IDs:")
for tok_id, count in top10:
    decoded = tokenizer.convert_ids_to_tokens([tok_id])[0]
    print(f"  id={tok_id:5d}  count={count:4d}  token='{decoded}'")

Total tokens in batch : 1024
Unique token IDs : 252

Top-10 most frequent token IDs:
  id= 1012  count=  71  token='.'
  id= 1996  count=  65  token='the'
  id= 1010  count=  46  token=','
  id= 1998  count=  42  token='and'
  id= 1037  count=  32  token='a'
  id= 2000  count=  26  token='to'
  id= 2001  count=  25  token='was'
  id= 2009  count=  15  token='it'
  id= 1999  count=  14  token='in'
  id= 1000  count=  14  token='"'


In [20]:
# 10. Decode a single block back to text
first_block_ids = sample_batch["input_ids"][0].tolist()
decoded_text = tokenizer.decode(first_block_ids, skip_special_tokens=True)
 
print("Decoded first block (128 tokens):")
print(decoded_text)


Decoded first block (128 tokens):
one day, a little girl named lily found a needle in her room. she knew it was difficult to play with it because it was sharp. lily wanted to share the needle with her mom, so she could sew a button on her shirt. lily went to her mom and said, " mom, i found this needle. can you share it with me and sew my shirt? " her mom smiled and said, " yes, lily, we can share the needle and fix your shirt. " together, they shared the needle and sewed the button on lily ' s shirt. it was not difficult for them because they were sharing


In [22]:
# 11. Raw text vs WordPiece tokens — side-by-side peek
# Shows exactly how bert-base-uncased splits words into subword pieces.
print("Raw text vs WordPiece tokenization (first 5 examples):\n")
peek_ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
for i, example in enumerate(peek_ds):
    raw_text = example["text"][:200]   # first 200 chars to keep output readable
    tokens   = tokenizer.tokenize(raw_text)
    print(f" Example {i+1} ")
    print(f"RAW : {raw_text}")
    print(f"TOKENS : {tokens}")
    print()
    if i == 4:
        break

Raw text vs WordPiece tokenization (first 5 examples):

 Example 1 
RAW : One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on
TOKENS : ['one', 'day', ',', 'a', 'little', 'girl', 'named', 'lily', 'found', 'a', 'needle', 'in', 'her', 'room', '.', 'she', 'knew', 'it', 'was', 'difficult', 'to', 'play', 'with', 'it', 'because', 'it', 'was', 'sharp', '.', 'lily', 'wanted', 'to', 'share', 'the', 'needle', 'with', 'her', 'mom', ',', 'so', 'she', 'could', 'se', '##w', 'a', 'button', 'on']

 Example 2 
RAW : Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, 
TOKENS : ['once', 'upon', 'a', 'time', ',', 'there', 'was', 'a', 'little', 'car', 'named', 'bee', '##p', '.', 'bee', '##p', 'loved', 'to', 'go', '

In [23]:
# 12. Vocabulary coverage
# What percentage of BERT's 30k vocabulary appears after streaming N batches?
N_BATCHES = 100
seen_ids  = set()
 
cov_loader = DataLoader(
    StreamingLMIterableDataset(
        load_dataset("roneneldan/TinyStories", split="train", streaming=True)
        .map(tokenize_function, batched=True),
        block_size
    ),
    batch_size=8,
    collate_fn=collate_fn
)
 
for i, batch in enumerate(cov_loader):
    seen_ids.update(batch["input_ids"].flatten().tolist())
    if i + 1 >= N_BATCHES:
        break
 
coverage = len(seen_ids) / tokenizer.vocab_size * 100
print(f"After {N_BATCHES} batches ({N_BATCHES * 8 * block_size:,} tokens):")
print(f" Unique token IDs seen : {len(seen_ids):,}")
print(f" Vocab coverage : {coverage:.2f}% of {tokenizer.vocab_size:,} BERT tokens")

After 100 batches (102,400 tokens):
 Unique token IDs seen : 3,559
 Vocab coverage : 11.66% of 30,522 BERT tokens


In [26]:
# 15. Validation split — same pipeline, compare batch stats with train
val_stream = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
val_tokenized = val_stream.map(tokenize_function, batched=True)
val_loader = DataLoader(
    StreamingLMIterableDataset(val_tokenized, block_size),
    batch_size=8,
    collate_fn=collate_fn
)
 
def batch_stats(loader, n_batches=20, label=""):
    all_ids = []
    for i, batch in enumerate(loader):
        all_ids.extend(batch["input_ids"].flatten().tolist())
        if i + 1 >= n_batches:
            break
    c = collections.Counter(all_ids)
    mean_id  = sum(all_ids) / len(all_ids)
    unique   = len(c)
    print(f"{label} ({n_batches} batches, {len(all_ids):,} tokens)")
    print(f" Unique token IDs : {unique}")
    print(f" Mean token ID : {mean_id:.1f}")
    print(f" Most common : id={c.most_common(1)[0][0]}  "
          f"token='{tokenizer.convert_ids_to_tokens([c.most_common(1)[0][0]])[0]}'")
    print()
 
print("Comparing train vs validation split stats:\n")
batch_stats(train_loader, label="Train")
batch_stats(val_loader, label="Validation")

Comparing train vs validation split stats:

Train (20 batches, 20,480 tokens)
 Unique token IDs : 1636
 Mean token ID : 3279.8
 Most common : id=1012  token='.'

Validation (20 batches, 20,480 tokens)
 Unique token IDs : 1499
 Mean token ID : 3288.9
 Most common : id=1012  token='.'

